In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score, f1_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import precision_recall_curve
import sys
# Get the current working directory of the notebook
current_dir = os.getcwd()
# Get the parent directory
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
# Add the parent directory to sys.path
sys.path.insert(0, parent_dir)
from imputation import * 
from internal_test import *
from summary_table import *
from feature_importance_COX import *
# Show all rows
pd.set_option('display.max_rows', None)
# Show all columns
pd.set_option('display.max_columns', None)
# Do not truncate column values
pd.set_option('display.max_colwidth', None)

In [2]:
# Define input columns and target
X_columns = [
    #'hhidpn', 'NIWWAVE','demcls',
     'child', 'lbrf', 'shlt', 'ageym', 'height', 'weight', 'smokev', 'effort', 'hibpe', 'diabe', 'vgactx', 'slfmem', 'livpar', 'momage', 'dadage', 'livsib', 'hlthlm', 'hosp', 'nrshom', 'nrstim', 'nrsnit', 'doctim', 'depres', 'sleepr', 'whappy', 'flone', 'fsad', 'going', 'enlife', 'drink', 'smoken', 'cancre', 'lunge', 'hearte', 'stroke', 'arthre', 'toilta', 'adl5a', 'mapa', 'walksa', 'walk1a', 'sita', 'chaira', 'climsa', 'clim1a', 'stoopa', 'lifta', 'dimea', 'armsa', 'pusha', 'mobila', 'lgmusa', 'grossa', 'finea', 'gender', 'edstg2 8to11', 'edstg3 12', 'edstg4 13above', 'cendiv2 mid atlantic', 'cendiv3 en central', 'cendiv4 wn central', 'cendiv5 s atlantic', 'cendiv6 es central', 'cendiv7 ws central', 'cendiv8 mountain', 'cendiv9 pacific', 'cendiv11 not us or inc us terr', 'mstat2 married spouse absent', 'mstat3 partnered', 'mstat4 separated', 'mstat5 divorced', 'mstat7 widowed', 'mstat8 never married', 'raceeth1 ', 'raceeth2 ', 'raceeth3 '
]

y_column = ['time','event']

In [3]:
train = pd.read_csv('../data/2000-2006/2000(00-06y)encode_COX.csv')
train.columns = train.columns.astype(str)
X_full_train, y_full_train = impute_data_and_y(train, 'mice', X_columns, y_column)
df_full = pd.merge(X_full_train, y_full_train, left_index=True, right_index=True)

**2000-2006**

In [4]:
test_2016 = pd.read_csv('../data/2000-2006/2010(10-16y)encode_COX.csv')
model = CoxPHFitter(penalizer=0.1, l1_ratio=0)
model.fit(df_full, duration_col='time', event_col='event')
X_test = test_2016[X_columns]
y_test = test_2016[y_column]
test_full = pd.merge(X_test, y_test, left_index=True, right_index=True)
threshold_time = 6
surv_probs = model.predict_survival_function(test_full).loc[threshold_time]
y_scores = 1 - surv_probs.values
optimal_threshold = 0.1982
y_pred = (y_scores >= optimal_threshold).astype(int)

auc = roc_auc_score(y_test['event'], y_scores)
acc = accuracy_score(y_test['event'], y_pred)
aupr = average_precision_score(y_test['event'], y_scores)
precision = precision_score(y_test['event'], y_pred)
recall = recall_score(y_test['event'], y_pred)
f1 = f1_score(y_test['event'], y_pred)

print("CoxPHFitter with MICE Imputation — External Test (2010-2016)")
print(f"Optimal Threshold Used: {optimal_threshold:.4f}")
print(
    f"Final test metrics: "
    f"AUC = {auc:.4f}, ACC = {acc:.4f}, AUPR = {aupr:.4f}, "
    f"Precision = {precision:.4f}, Recall = {recall:.4f}, F1 = {f1:.4f}"
)

CoxPHFitter with MICE Imputation — External Test (2010-2016)
Optimal Threshold Used: 0.1982
Final test metrics: AUC = 0.7878, ACC = 0.9063, AUPR = 0.2633, Precision = 0.3671, Recall = 0.1800, F1 = 0.2415


**2000-2008**

In [ ]:
test_2018 = pd.read_csv('../data/2000-2008/2010(10-18y)encode_COX.csv')
model = CoxPHFitter(penalizer=0.1, l1_ratio=0)
model.fit(df_full, duration_col='time', event_col='event')
X_test = test_2018[X_columns]
y_test = test_2018[y_column]
test_full = pd.merge(X_test, y_test, left_index=True, right_index=True)
threshold_time = 6
surv_probs = model.predict_survival_function(test_full).loc[threshold_time]
y_scores = 1 - surv_probs.values
optimal_threshold = 0.1982
y_pred = (y_scores >= optimal_threshold).astype(int)

auc = roc_auc_score(y_test['event'], y_scores)
acc = accuracy_score(y_test['event'], y_pred)
aupr = average_precision_score(y_test['event'], y_scores)
precision = precision_score(y_test['event'], y_pred)
recall = recall_score(y_test['event'], y_pred)
f1 = f1_score(y_test['event'], y_pred)

print("CoxPHFitter with MICE Imputation — External Test (2010-2018)")
print(f"Optimal Threshold Used: {optimal_threshold:.4f}")
print(
    f"Final test metrics: "
    f"AUC = {auc:.4f}, ACC = {acc:.4f}, AUPR = {aupr:.4f}, "
    f"Precision = {precision:.4f}, Recall = {recall:.4f}, F1 = {f1:.4f}"
)

CoxPHFitter with MICE Imputation — External Test (2010-2018)
Optimal Threshold Used: 0.1982
Final test metrics: AUC = 0.7866, ACC = 0.8954, AUPR = 0.2855, Precision = 0.3928, Recall = 0.1664, F1 = 0.2338
